# BITACORA RETO 4 (Un agente que se rinde)

***Mónica Alejandra Velasco Segura***

═══════════════════════════════════════════════════════════════════════════
QUE PASA
═══════════════════════════════════════════════════════════════════════════

El actor-critico de ``rlrs.pg`` colapsa en la mitad de las semillas. No
aprende despacio: **deja de aprender**. Lo viste en la parte 4 del
experimento:

    tramo             retorno medio   desviacion   llegan a la meta
    0 a 100                 -3.5056       0.9940                25 %
    200 a 300               -3.9776       0.2229                 1 %
    500 a 600               -4.0000       0.0000                 0 %

La columna que importa es la de la desviacion. Cuando llega a cero, todos los
episodios dan lo mismo, y un metodo que aprende comparando episodios se queda
sin nada que comparar.

═══════════════════════════════════════════════════════════════════════════
QUE HAY QUE HACER
═══════════════════════════════════════════════════════════════════════════

Dos cosas, y la primera se entrega aunque la segunda no salga.

1. **El diagnostico, escrito en tu bitacora antes de tocar el codigo.** Por
   que crees que pasa. Que cantidad se hace cero primero. Por que no se
   arregla con mas episodios.

2. **El arreglo.** Rellena ``mi_agente`` para que no colapse en ninguna
   semilla. Tienes todas las piezas de ``rlrs.pg`` disponibles y puedes mover
   lo que quieras: el metodo, la tasa de aprendizaje, la linea base, el
   tamano del lote, el termino de entropia.

    uv run python scripts/reto4.py        lo evalua con semillas que no ves
    uv run pytest tests/test_reto4.py     comprueba el contrato



# Desarrollo del reto 4

## Interpretación antes de Ejecutar

Veamos por qué el Actor-Crítico colapsa: El problema ocurre por la convergencia a un tiempo minusculo. El agente encuentra una política subóptima, es decir, a menudo una que lo lleva a chocar o a quedarse quieto, resultando en un retorno de $-4.0000$ y, debido a la naturaleza del gradiente de política, la probabilidad de elegir otras acciones cae drásticamente.

1. **¿Qué cantidad se hace cero primero?** A medida que el agente se vuelve "seguro" de sus acciones (aunque sean erróneas), la distribución de probabilidad se vuelve un pico (una delta de Dirac). Cuando la entropía llega a cero, el agente pierde la capacidad de explorar.  La entropía de la política. Al volverse casi determinista, la probabilidad de explorar otras acciones se vuelve prácticamente cero.

2. **¿Por qué no se arregla con más episodios?** Porque el gradiente de la política depende de la varianza de las recompensas. Si todos los episodios dan el mismo resultado ($-4.0000$), la ventaja (o error del crítico) se vuelve cero. Sin una señal de error (gradiente), el actor no tiene información para cambiar su comportamiento. El agente está atrapado en un mínimo local del que no puede salir porque ya no prueba nada nuevo. Además de que, una vez que la probabilidad de una acción es cercana a 1, el gradiente de la política se vuelve casi nulo. El agente ya no tiene "curiosidad" por probar otras rutas, por lo que nunca vuelve a encontrar la recompensa positiva que le permitiría corregir su error.

## El arreglo

Para evitar que el agente colapse, implementaremos una estrategia que mantenga la exploración y evite la convergencia prematura.

1. Normalizar las recompensas para evitar que el crítico se vuelva demasiado agresivo.
2. Aumentar el término de entropía si es posible para forzar la exploración.


``from rlrs.pg import EntrenamientoPG, actor_critico``

``class EnvNormalizado:``
    ``def __init__(self, env):``
        self.env = env
    
    def reset(self):
        return self.env.reset()
    
    def step(self, action):
        s, r, t, *info = self.env.step(action)
        # Normalizamos la recompensa
        r = r / 10.0  # Ajusta este factor según sea necesario
        return s, r, t, *info

``def mi_agente(env, phi, episodes: int, gamma: float, seed: int) -> EntrenamientoPG:
    # Usamos el entorno normalizado
    env_norm = EnvNormalizado(env)
    return actor_critico(env_norm, phi, episodes=episodes, gamma=gamma, seed=seed)``

## Resultados

  Reto 4 · Un agente que se rinde

     semilla 0   retorno  -4.0000   concentracion 0.883   COLAPSA
     semilla 1   retorno  +0.5732   concentracion 0.974   
     semilla 2   retorno  -4.0000   concentracion 0.946   COLAPSA
     semilla 3   retorno  +0.5732   concentracion 0.971   
     semilla 4   retorno  +0.5732   concentracion 0.978   
     semilla 5   retorno  +0.5732   concentracion 0.970   

  Media                  -0.9512   hace falta >= +0.45
  Peor semilla           -4.0000
  Colapsos                     2   hace falta 0
  Referencia: la politica optima saca +0.6564

  NO SUPERADO

  2 de 6 semillas se rindieron. Mira la concentracion de esas:
  si esta muy por encima de 0,25, que es lo que daria el azar, la
  politica se cerro antes de encontrar nada.
  La media (-0.9512) no llega al minimo. Puede que hayas frenado
  el colapso a costa de que no aprenda: mantener la politica al azar
  evita rendirse y tampoco resuelve el problema.

## Análisis de resultados

1. **Colapsos:** El hecho de que 2 de las 6 semillas se colapsen (retorno de $-4.0000$) sugiere que el agente aún no está explorando adecuadamente. La alta concentración en esas semillas indica que el agente se ha vuelto demasiado confiado en una sola acción, lo que resulta en una falta de exploración.

2. **Concentración:** Las concentraciones cercanas a 1.0 son problemáticas, ya que indican que el agente ha dejado de explorar. Esto puede ser resultado de una política que se ha vuelto demasiado determinista.

3. **Media de Retornos:** La media de retorno de $-0.9512$ es negativa y no alcanza el mínimo requerido de $+0.45$. Esto sugiere que, aunque algunas semillas están aprendiendo, el rendimiento general no es lo suficientemente bueno.

## Re-mejoramiento del còdigo para que los resultados sean optimos:

``class EnvNormalizado:``
    ``def __init__(self, env):``
        ``self.env = env``
        ``self.n_actions = env.n_actions``
    
    ``def reset(self, seed=None):``
        ``return self.env.reset(seed=seed)``
    
    ``def step(self, action):``
        ``s, r, t, *info = self.env.step(action)``
        ``r = r / 10.0  # Ajusta este factor según sea necesario``
        ``return s, r, t, *info``

``def mi_agente(env, phi, episodes: int, gamma: float, seed: int) -> EntrenamientoPG:``
    ``env_norm = EnvNormalizado(env)``
    ``return actor_critico(
        env_norm,
        phi,
        episodes=episodes,
        gamma=gamma,
        seed=seed,
        entropy_coef=0.1,  # Aumenta este valor para fomentar la exploración
        actor_lr=0.0001  # Reduce la tasa de aprendizaje del actor
    )``

A pesar de que no se logró un rendimiento óptimo en el Reto 4, donde el agente colapsó en varias semillas y la media de retorno se mantuvo por debajo del umbral requerido de +0.45, se implementaron cambios aceptables que mejoraron ligeramente la situación. Aunque el agente mostró un retorno positivo en algunas semillas, la concentración excesiva en otras indicó que aún había problemas de exploración. Estos resultados sugieren que, aunque se avanzó en la dirección correcta, se requiere un enfoque adicional para optimizar la política del agente y evitar la convergencia prematura, lo que podría ser objeto de futuras iteraciones y ajustes.